# Sanofi India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** sanofi.wd3.myworkdayjobs.com/SanofiCareers

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills


Imports loaded. Date: 2026-03-27 18:21:06
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Sanofi"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Sanofi/Outputs/2026_03_27


In [4]:
print("=" * 60)
print("SANOFI INDIA JOB SCRAPER")
print("ATS: Workday (sanofi.wd3.myworkdayjobs.com)")
print("=" * 60)

sanofi_jobs = scrape_workday(
    tenant="sanofi",
    instance="wd3",
    career_site="SanofiCareers",
    company_name="Sanofi",
    industry="Pharmaceutical",
    location_filter=LOCATION_FILTER,
    max_jobs=500
)

# Fallback: also try jobs.sanofi.com if Workday returns few results
if len(sanofi_jobs) < 5:
    print("\n  Few results from Workday, trying jobs.sanofi.com fallback...")
    try:
        session = get_session()
        base = "https://jobs.sanofi.com/en/search-jobs/India"
        resp = session.get(base, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            cards = soup.select('#search-results-list a[href*="/en/job/"]')
            for card in cards:
                title_el = card.select_one("h2, h3, strong")
                if title_el:
                    title = title_el.get_text(strip=True)
                    href = card.get("href", "")
                    job_id = href.split("/")[-1] if href else str(len(sanofi_jobs))
                    sanofi_jobs.append({
                        "job_id": job_id,
                        "title": title,
                        "company_name": "Sanofi",
                        "raw_jd_text": card.get_text(" ", strip=True),
                        "location_city": "India",
                        "industry": "Pharmaceutical",
                        "date_posted": datetime.now().strftime("%Y-%m-%d"),
                        "is_active": True,
                        "job_url": href if href.startswith("http") else f"https://jobs.sanofi.com{href}",
                        "business_unit": "",
                        "source_platform": "Sanofi fallback",
                    })
            print(f"  Fallback found {len(cards)} additional jobs")
    except Exception as e:
        print(f"  Fallback failed: {e}")


SANOFI INDIA JOB SCRAPER
ATS: Workday (sanofi.wd3.myworkdayjobs.com)
  Scraping Sanofi via Workday API: https://sanofi.wd3.myworkdayjobs.com/wday/cxs/sanofi/SanofiCareers/jobs
  Mode: BROAD (no location filter — fetching all global jobs)


  Total India jobs reported by API: 1223
  Page offset=0: 20 jobs (page_total=1223, known_total=1223)


  Page offset=20: 20 jobs (page_total=0, known_total=1223)


  Page offset=40: 20 jobs (page_total=0, known_total=1223)


  Page offset=60: 20 jobs (page_total=0, known_total=1223)


  Page offset=80: 20 jobs (page_total=0, known_total=1223)


  [ERROR] HTTPSConnectionPool(host='sanofi.wd3.myworkdayjobs.com', port=443): Read timed out. (read timeout=30)
  Total Sanofi India jobs: 100


In [5]:
df_sanofi = save_results(sanofi_jobs, "Sanofi", OUTPUT_DIR)
if df_sanofi is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_sanofi.columns]
    print(df_sanofi[cols].head(10).to_string())


  [OK] Saved 100 jobs -> Sanofi_jobs_2026-03-27.csv
       Seniority: {'junior': 52, 'lead': 36, 'mid': 12}
       Work mode: {'onsite': 78, 'hybrid': 17, 'remote': 5}
       Has JD text: 100/100
       Has job URL: 100/100
       Has business unit: 0/100

Sample jobs:
                                                                       title      location_city seniority_level business_unit                                                                                                                                                             job_url
0                           Project Manager - Omnichannel Program Excellence             Bogota          junior                                                              https://sanofi.wd3.myworkdayjobs.com/en-US/SanofiCareers//job/Bogota/Omnichannel-Program-Excellence-Manager_R2842604
1                                            Scientist, Retinal Degeneration      Cambridge, MA          junior                                       